# GroupBy and Aggregations

So far, our transformations have worked on one row at a time. Aggregations answer questions about groups of rows, such as total revenue by region.

---
## Learning objectives

By the end of this notebook, you will be able to:

- group rows with `groupBy()`;
- calculate totals, averages, and counts with `agg()`; and
- group by one or more columns.

---
## Set up the sales data

This is a small, self-contained sales DataFrame. Each row represents one order and already contains its order value.

In [ ]:
from pyspark.sql import functions as F

sales_rows = [
    (1001, "North", "Stationery", 37.00),
    (1002, "West", "Furniture", 750.00),
    (1003, "North", "Stationery", 36.00),
    (1004, "North", "Stationery", 92.50),
    (1005, "South", "Electronics", 170.00),
    (1006, "West", "Stationery", 48.00),
    (1007, "North", "Furniture", 750.00),
    (1008, "South", "Stationery", 55.50),
]

sales_schema = """
    order_id INT,
    region STRING,
    category STRING,
    order_value DOUBLE
"""

retail_sales = spark.createDataFrame(sales_rows, schema=sales_schema)
retail_sales.show()

---
## Group rows, then calculate

`groupBy("region")` creates one group for every region. To produce a result, we then apply a calculation to each group. This changes the grain of the data: the result has one row per region rather than one row per order.

Start with a single, simple calculation before combining several calculations with `agg()`.

### Count rows in each group

`count()` is a shortcut for one common aggregation. It counts the order rows in each region.

In [ ]:
order_count_by_region = retail_sales.groupBy("region").count()
order_count_by_region.show()

### Sum one column in each group

`sum("order_value")` is another shortcut. It adds the order value for each region.

In [ ]:
total_revenue_by_region = retail_sales.groupBy("region").sum("order_value")
total_revenue_by_region.show()

---
## Combine measures with `agg()`

Shortcuts are clear when we need one simple calculation. Use `agg()` when we need several calculations together or want to give calculated columns meaningful names.

In [ ]:
revenue_by_region = retail_sales.groupBy("region").agg(
    F.sum("order_value").alias("total_revenue"),
    F.count("*").alias("order_count"),
    F.avg("order_value").alias("average_order_value"),
)

revenue_by_region.show()

`F.sum()` adds values, `F.count()` counts rows, and `F.avg()` calculates an average. `alias()` gives each calculated column a useful output name. Other common aggregate functions are `F.min()` and `F.max()`.

---
## Group by more than one column

Passing multiple column names creates a group for each unique combination. This result has one row for every region and category pair.

In [ ]:
revenue_by_region_category = retail_sales.groupBy(
    "region", "category"
).agg(
    F.sum("order_value").alias("total_revenue"),
    F.count("*").alias("order_count"),
)

revenue_by_region_category.show()

---
## Your turn

**Exercise 1:** How many orders did each product category receive? Create a DataFrame named `order_count_by_category` with one row per category and its order count. Preview it with `show()`.

In [ ]:
# Write your solution here.

**Exercise 2:** Create a DataFrame named `order_value_range_by_region`. For each region, calculate the smallest and largest order value, naming the output columns `smallest_order_value` and `largest_order_value`. Preview it with `show()`. *Clue: the functions are "other common aggregate functions"*

In [ ]:
# Write your solution here.

---
## Next lesson

Next, we will combine related DataFrames with joins.